In [1]:

# ============================================================
# JEEVAN LINK: SOS URGENCY DATASET BUILDER
# ============================================================
# This notebook prepares a large dataset for the later training
# stage. Generated scenarios and their labels are synthetic.
#
# IMPORTANT:
# - Source flood data is contextual, not verified SOS data.
# - IFI district tables are kept separate to avoid unsafe joins.
# - INDOFLOODS events and precipitation are joined by EventID.
# - This is a project-development dataset, not a validated
#   real-world emergency triage dataset.
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

# ------------------------------------------------------------
# 1. Configure paths
# ------------------------------------------------------------

# Expected location:
# Data/Processed_data/sos_urgency_dataset/
#     02_sos_urgency_dataset_builder.ipynb

BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR.parent

SOS_FILE = BASE_DIR / "sos_urgency_data_collection.csv"

IFI_DIR = DATA_DIR / "ifi_processed_Data"
INDO_DIR = DATA_DIR / "indoflood_processed_Data"
DR_DIR = DATA_DIR / "Disaster Response Messages"

OUTPUT_FILE = BASE_DIR / "sos_urgency_dataset_processed.csv"

required_files = {
    "SOS collection": SOS_FILE,
    "IFI impact": IFI_DIR / "District_FloodImpact_processed.csv",
    "IFI flooded area": IFI_DIR / "District_FloodedArea_processed.csv",
    "IFI severity": IFI_DIR / "DFSI_processed.csv",
    "INDOFLOODS events": INDO_DIR / "floodevents_indofloods_processed.csv",
    "INDOFLOODS precipitation": INDO_DIR / "precipitation_variables_indofloods_processed.csv",
    "Disaster Response": DR_DIR / "disaster_response_training_processed.csv",
}

missing_files = [
    f"{name}: {path}"
    for name, path in required_files.items()
    if not path.exists()
]

if missing_files:
    raise FileNotFoundError(
        "Some input files were not found.\n"
        "Check your folder structure and filenames:\n"
        + "\n".join(missing_files)
    )

print("All required input files were found.")


# ------------------------------------------------------------
# 2. Load the processed source datasets
# ------------------------------------------------------------

sos_original = pd.read_csv(SOS_FILE, low_memory=False)

ifi_impact = pd.read_csv(
    required_files["IFI impact"], low_memory=False
)

ifi_area = pd.read_csv(
    required_files["IFI flooded area"], low_memory=False
)

ifi_severity = pd.read_csv(
    required_files["IFI severity"], low_memory=False
)

flood_events = pd.read_csv(
    required_files["INDOFLOODS events"], low_memory=False
)

precipitation = pd.read_csv(
    required_files["INDOFLOODS precipitation"], low_memory=False
)

disaster_messages = pd.read_csv(
    required_files["Disaster Response"], low_memory=False
)

print("\nSOURCE DATASET SIZES")
print("Existing SOS records:", len(sos_original))
print("IFI impact records:", len(ifi_impact))
print("IFI flooded-area records:", len(ifi_area))
print("IFI severity records:", len(ifi_severity))
print("INDOFLOODS event records:", len(flood_events))
print("INDOFLOODS precipitation records:", len(precipitation))
print("Disaster Response messages:", len(disaster_messages))


# ------------------------------------------------------------
# 3. Validate required columns
# ------------------------------------------------------------

required_sos_columns = [
    "sos_id",
    "sos_description",
    "people_affected",
    "people_trapped",
    "injuries_reported",
    "medical_emergency",
    "water_level",
    "building_condition",
    "rescue_required",
    "food_required",
    "water_required",
    "shelter_required",
    "location_risk",
    "urgency_label",
]

missing_sos_columns = [
    c for c in required_sos_columns
    if c not in sos_original.columns
]

if missing_sos_columns:
    raise ValueError(
        f"SOS collection is missing columns: {missing_sos_columns}"
    )

if "EventID" not in flood_events.columns:
    raise ValueError("EventID is missing from the INDOFLOODS event file.")

if "EventID" not in precipitation.columns:
    raise ValueError("EventID is missing from the precipitation file.")

if "message" not in disaster_messages.columns:
    raise ValueError("The Disaster Response file has no message column.")


# ------------------------------------------------------------
# 4. Join INDOFLOODS using EventID
# ------------------------------------------------------------

# Remove duplicate precipitation EventIDs before joining.
# This avoids multiplying event rows during the merge.
precipitation_unique = precipitation.drop_duplicates(
    subset=["EventID"],
    keep="first"
)

flood_context = flood_events.merge(
    precipitation_unique,
    on="EventID",
    how="left",
    suffixes=("_event", "_rain"),
    validate="one_to_one"
)

print("\nINDOFLOODS event/context rows:", len(flood_context))


# ------------------------------------------------------------
# 5. Keep IFI tables separate
# ------------------------------------------------------------

# District names repeat in IFI, so merging on Dist_Name alone
# could combine unrelated districts or multiply records.
# Keep the tables separate until a reliable common key is known.

ifi_impact_context = ifi_impact.copy()
ifi_area_context = ifi_area.copy()
ifi_severity_context = ifi_severity.copy()

print("\nIFI tables retained separately:")
print("Impact table:", ifi_impact_context.shape)
print("Flooded-area table:", ifi_area_context.shape)
print("Severity table:", ifi_severity_context.shape)


# ------------------------------------------------------------
# 6. Prepare helper functions
# ------------------------------------------------------------

rng = np.random.default_rng(42)

def sample_row(dataframe):
    """Select one source row for illustrative contextual sampling."""
    if dataframe.empty:
        raise ValueError("A source dataset is empty; cannot sample from it.")
    return dataframe.iloc[int(rng.integers(0, len(dataframe)))]

def safe_number(row, column, default=np.nan):
    """Read a numeric source value safely."""
    if column not in row.index:
        return default

    value = pd.to_numeric(
        pd.Series([row[column]]),
        errors="coerce"
    ).iloc[0]

    return float(value) if pd.notna(value) else default

def yes_no(condition):
    """Convert a condition into a consistent Yes/No value."""
    return "Yes" if bool(condition) else "No"


# Select assistance labels available in the Disaster Response data.
assistance_columns = [
    "medical_help",
    "search_and_rescue",
    "water",
    "food",
    "shelter",
    "clothing",
    "transport",
    "buildings",
    "floods",
]

assistance_columns = [
    column for column in assistance_columns
    if column in disaster_messages.columns
]


# ------------------------------------------------------------
# 7. Define illustrative urgency rules
# ------------------------------------------------------------

def assign_synthetic_urgency(row):
    """
    Example-only rules for generated scenarios.
    They are not a clinically or operationally validated
    emergency triage standard.
    """

    if (
        row["medical_emergency"] == "Yes"
        or (
            row["people_trapped"] == "Yes"
            and row["water_level"] == "Rapidly_Rising"
        )
        or (
            row["building_condition"] == "Collapsing"
            and row["people_trapped"] == "Yes"
        )
    ):
        return "Critical"

    if (
        row["people_trapped"] == "Yes"
        or row["injuries_reported"] == "Yes"
        or row["rescue_required"] == "Yes"
        or row["location_risk"] == "High"
    ):
        return "High"

    if (
        row["location_risk"] == "Medium"
        or row["food_required"] == "Yes"
        or row["water_required"] == "Yes"
        or row["shelter_required"] == "Yes"
        or row["building_condition"] == "Damaged"
    ):
        return "Medium"

    return "Low"


# ------------------------------------------------------------
# 8. Generate additional synthetic SOS scenarios
# ------------------------------------------------------------

N_SYNTHETIC = 5000

water_levels = [
    "Stable",
    "Rising",
    "Rapidly_Rising",
]

building_conditions = [
    "Intact",
    "Damaged",
    "Collapsing",
    "Unknown",
]

location_risks = [
    "Low",
    "Medium",
    "High",
]

synthetic_records = []

for i in range(N_SYNTHETIC):

    # These are independent context samples.
    # They are NOT confirmed to describe the same real location.
    event_row = sample_row(flood_context)
    impact_row = sample_row(ifi_impact_context)
    area_row = sample_row(ifi_area_context)
    severity_row = sample_row(ifi_severity_context)
    message_row = sample_row(disaster_messages)

    # Generate illustrative SOS-level features.
    people_affected = int(rng.integers(1, 31))

    people_trapped = rng.choice(
        ["Yes", "No"],
        p=[0.25, 0.75]
    )

    injuries_reported = rng.choice(
        ["Yes", "No"],
        p=[0.30, 0.70]
    )

    medical_emergency = rng.choice(
        ["Yes", "No"],
        p=[0.18, 0.82]
    )

    water_level = rng.choice(
        water_levels,
        p=[0.30, 0.45, 0.25]
    )

    building_condition = rng.choice(
        building_conditions,
        p=[0.55, 0.25, 0.05, 0.15]
    )

    location_risk = rng.choice(
        location_risks,
        p=[0.25, 0.40, 0.35]
    )

    # Read assistance categories from a sampled Disaster Response row.
    # These categories are contextual examples, not verified needs
    # for the generated SOS scenario.
    assistance_values = {}

    for column in assistance_columns:
        value = pd.to_numeric(
            pd.Series([message_row[column]]),
            errors="coerce"
        ).iloc[0]

        assistance_values[column] = (
            int(value) if pd.notna(value) else np.nan
        )

    # Use SOS features for generated resource requests.
    rescue_required = yes_no(
        people_trapped == "Yes"
        or injuries_reported == "Yes"
        or medical_emergency == "Yes"
    )

    food_required = yes_no(
        rng.random() < 0.30
    )

    water_required = yes_no(
        rng.random() < 0.35
    )

    shelter_required = yes_no(
        building_condition in ["Damaged", "Collapsing"]
        or rng.random() < 0.20
    )

    # Construct the scenario.
    record = {
        "sos_id": f"SYN{i + 1:06d}",

        "sos_description": (
            f"Synthetic flood scenario affecting {people_affected} people. "
            f"Water level: {water_level}. "
            f"Building condition: {building_condition}. "
            f"Location risk: {location_risk}."
        ),

        "people_affected": people_affected,
        "people_trapped": people_trapped,
        "injuries_reported": injuries_reported,
        "medical_emergency": medical_emergency,
        "water_level": water_level,
        "building_condition": building_condition,
        "rescue_required": rescue_required,
        "food_required": food_required,
        "water_required": water_required,
        "shelter_required": shelter_required,
        "location_risk": location_risk,

        # INDOFLOODS context.
        "indoflood_event_id_context": event_row.get(
            "EventID", np.nan
        ),
        "context_peak_flood_level_m": safe_number(
            event_row, "Peak Flood Level (m)"
        ),
        "context_event_duration_days": safe_number(
            event_row, "Event Duration (days)"
        ),
        "context_rainfall_t1d": safe_number(
            event_row, "T1d"
        ),

        # IFI context.
        "ifi_impact_district_context": impact_row.get(
            "Dist_Name", np.nan
        ),
        "context_district_fatalities": safe_number(
            impact_row, "Human_fatality"
        ),
        "context_district_injuries": safe_number(
            impact_row, "Human_injured"
        ),
        "context_district_population": safe_number(
            impact_row, "Population"
        ),

        "ifi_area_district_context": area_row.get(
            "Dist_Name", np.nan
        ),
        "context_percent_flooded_area": safe_number(
            area_row, "Corrected_Percent_Flooded_Area",
            default=safe_number(area_row, "Percent_Flooded_Area")
        ),

        "ifi_severity_district_context": severity_row.get(
            "District_Name", np.nan
        ),
        "context_dfsi": safe_number(severity_row, "DFSI"),

        # Disaster Response context.
        "disaster_response_message_id_context": message_row.get(
            "id", np.nan
        ),
        "disaster_response_message_context": str(
            message_row["message"]
        ),

        # Retain available assistance label columns with a source_ prefix.
        **{
            f"source_{column}": value
            for column, value in assistance_values.items()
        },

        # Provenance.
        "record_source": "synthetic",
        "label_source": "synthetic_rule",
        "context_link_status": "independent_sample_not_linked",
    }

    record["urgency_label"] = assign_synthetic_urgency(record)
    synthetic_records.append(record)

synthetic_df = pd.DataFrame(synthetic_records)

print("\nSynthetic SOS scenarios created:", len(synthetic_df))


# ------------------------------------------------------------
# 9. Prepare the original SOS records
# ------------------------------------------------------------

# Preserve original SOS records and existing labels.
# Their labels have not been independently verified.
original_df = sos_original.copy()

if "record_source" not in original_df.columns:
    original_df["record_source"] = "unknown"

if "label_source" not in original_df.columns:
    original_df["label_source"] = "existing_label_unverified"

if "context_link_status" not in original_df.columns:
    original_df["context_link_status"] = "not_recorded"

# Add any additional generated-dataset columns to original records
# as missing, since no reliable contextual linkage has been confirmed.
for column in synthetic_df.columns:
    if column not in original_df.columns:
        original_df[column] = np.nan

for column in original_df.columns:
    if column not in synthetic_df.columns:
        synthetic_df[column] = np.nan

# Use the same column order in both dataframes.
synthetic_df = synthetic_df[original_df.columns]

# Combine original records first, followed by synthetic scenarios.
final_df = pd.concat(
    [original_df, synthetic_df],
    ignore_index=True
)

# Remove duplicate IDs only; retain the first occurrence.
# Synthetic IDs use a separate SYN prefix.
final_df = final_df.drop_duplicates(
    subset=["sos_id"],
    keep="first"
).reset_index(drop=True)


# ------------------------------------------------------------
# 10. Validate the final dataset
# ------------------------------------------------------------

valid_labels = [
    "Low",
    "Medium",
    "High",
    "Critical",
]

print("\n" + "=" * 50)
print("FINAL SOS URGENCY DATASET REPORT")
print("=" * 50)

print("Total rows:", len(final_df))
print("Total columns:", len(final_df.columns))

print(
    "Duplicate SOS IDs:",
    final_df["sos_id"].duplicated().sum()
)

print(
    "Missing SOS IDs:",
    final_df["sos_id"].isna().sum()
)

print(
    "Invalid urgency labels:",
    (~final_df["urgency_label"].isin(valid_labels)).sum()
)

print("\nRecords by source:")
display(
    final_df["record_source"].value_counts(dropna=False)
)

print("\nLabel-source distribution:")
display(
    final_df["label_source"].value_counts(dropna=False)
)

print("\nUrgency-label distribution:")
display(
    final_df["urgency_label"]
    .value_counts(dropna=False)
    .reindex(valid_labels, fill_value=0)
)

print("\nMissing values in core columns:")
core_columns = [
    "sos_id",
    "sos_description",
    "people_affected",
    "urgency_label",
    "record_source",
    "label_source",
]

display(
    final_df[core_columns]
    .isna()
    .sum()
    .to_frame("Missing_Count")
)

print("\nDataset preview:")
display(final_df.head(10))


# ------------------------------------------------------------
# 11. Save the processed dataset
# ------------------------------------------------------------

# Save a new output file. The collection CSV is not overwritten.
final_df.to_csv(
    OUTPUT_FILE,
    index=False
)

print("\nDataset successfully saved to:")
print(OUTPUT_FILE.resolve())

print("\nOriginal collection file was not overwritten.")
print("Model training has NOT started.")


All required input files were found.

SOURCE DATASET SIZES
Existing SOS records: 16
IFI impact records: 732
IFI flooded-area records: 732
IFI severity records: 744
INDOFLOODS event records: 4548
INDOFLOODS precipitation records: 4548
Disaster Response messages: 19099

INDOFLOODS event/context rows: 4548

IFI tables retained separately:
Impact table: (732, 5)
Flooded-area table: (732, 4)
Severity table: (744, 3)

Synthetic SOS scenarios created: 5000

FINAL SOS URGENCY DATASET REPORT
Total rows: 5016
Total columns: 40
Duplicate SOS IDs: 0
Missing SOS IDs: 0
Invalid urgency labels: 0

Records by source:


record_source
synthetic    5016
Name: count, dtype: int64


Label-source distribution:


label_source
synthetic_rule               5000
existing_label_unverified      16
Name: count, dtype: int64


Urgency-label distribution:


urgency_label
Low          132
Medium      1305
High        2375
Critical    1204
Name: count, dtype: int64


Missing values in core columns:


,Missing_Count
sos_id,0
sos_description,0
people_affected,0
urgency_label,0
record_source,0
label_source,0



Dataset preview:


,sos_id,sos_description,people_affected,people_trapped,injuries_reported,medical_emergency,water_level,building_condition,rescue_required,food_required,...,disaster_response_message_context,source_medical_help,source_search_and_rescue,source_water,source_food,source_shelter,source_clothing,source_transport,source_buildings,source_floods
0,SOS0001,Four people trapped on a roof as water rises r...,4,Yes,No,No,Rapidly_Rising,Intact,Yes,No,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,SOS0002,An elderly person is unconscious in a flooded ...,1,Yes,Yes,Yes,Rising,Damaged,Yes,No,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,SOS0003,A child is trapped in fast-moving floodwater.,1,Yes,No,No,Rapidly_Rising,Unknown,Yes,No,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,SOS0004,Several people are trapped inside a collapsing...,6,Yes,Yes,No,Rising,Damaged,Yes,No,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,SOS0005,Water is entering the ground floor and a famil...,5,No,No,No,Rising,Intact,Yes,No,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,SOS0006,An injured resident needs urgent transport to ...,1,No,Yes,Yes,Stable,Intact,Yes,No,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,SOS0007,A family is stranded by floodwater with no saf...,4,No,No,No,Rising,Intact,Yes,No,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,SOS0008,A damaged house may not be safe to occupy duri...,3,No,No,No,Stable,Damaged,No,No,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,SOS0009,A family needs drinking water after the flood ...,4,No,No,No,Stable,Intact,No,No,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,SOS0010,Two residents need dry blankets and temporary ...,2,No,No,No,Stable,Intact,No,No,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



Dataset successfully saved to:
C:\Users\ishit\jeevan_link_ml\Data\Processed_data\sos_urgency_dataset\sos_urgency_dataset_processed.csv

Original collection file was not overwritten.
Model training has NOT started.


In [2]:

# STEP 4: Inspect and validate the saved processed dataset

from pathlib import Path
import pandas as pd

# Locate the already-saved dataset
output_file = Path.cwd() / "sos_urgency_dataset_processed.csv"

if not output_file.exists():
    raise FileNotFoundError(
        f"Processed dataset not found: {output_file}"
    )

# Load the saved dataset
final_df = pd.read_csv(output_file, low_memory=False)

print("Dataset file:", output_file.resolve())
print("Dataset shape:", final_df.shape)

# Check the source of the records
print("\nRecord source counts:")
display(final_df["record_source"].value_counts(dropna=False))

# Check urgency labels
print("\nUrgency label counts:")
display(final_df["urgency_label"].value_counts(dropna=False))

# Check duplicate IDs
print("\nDuplicate SOS IDs:", final_df["sos_id"].duplicated().sum())

# Check missing values in core columns
core_columns = [
    "sos_id",
    "sos_description",
    "people_affected",
    "urgency_label",
    "record_source",
]

print("\nMissing values in core columns:")
display(final_df[core_columns].isna().sum().to_frame("Missing_Count"))

# Display a few records
print("\nFirst five records:")
display(final_df.head())

# Confirm the source and label provenance columns
print("\nAvailable provenance columns:")
print([
    column for column in final_df.columns
    if "source" in column.lower()
    or "label" in column.lower()
    or "link_status" in column.lower()
])


Dataset file: C:\Users\ishit\jeevan_link_ml\Data\Processed_data\sos_urgency_dataset\sos_urgency_dataset_processed.csv
Dataset shape: (5016, 40)

Record source counts:


record_source
synthetic    5016
Name: count, dtype: int64


Urgency label counts:


urgency_label
High        2375
Medium      1305
Critical    1204
Low          132
Name: count, dtype: int64


Duplicate SOS IDs: 0

Missing values in core columns:


,Missing_Count
sos_id,0
sos_description,0
people_affected,0
urgency_label,0
record_source,0



First five records:


,sos_id,sos_description,people_affected,people_trapped,injuries_reported,medical_emergency,water_level,building_condition,rescue_required,food_required,...,disaster_response_message_context,source_medical_help,source_search_and_rescue,source_water,source_food,source_shelter,source_clothing,source_transport,source_buildings,source_floods
0,SOS0001,Four people trapped on a roof as water rises r...,4,Yes,No,No,Rapidly_Rising,Intact,Yes,No,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,SOS0002,An elderly person is unconscious in a flooded ...,1,Yes,Yes,Yes,Rising,Damaged,Yes,No,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,SOS0003,A child is trapped in fast-moving floodwater.,1,Yes,No,No,Rapidly_Rising,Unknown,Yes,No,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,SOS0004,Several people are trapped inside a collapsing...,6,Yes,Yes,No,Rising,Damaged,Yes,No,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,SOS0005,Water is entering the ground floor and a famil...,5,No,No,No,Rising,Intact,Yes,No,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



Available provenance columns:
['urgency_label', 'record_source', 'label_source', 'context_link_status', 'source_medical_help', 'source_search_and_rescue', 'source_water', 'source_food', 'source_shelter', 'source_clothing', 'source_transport', 'source_buildings', 'source_floods']


In [3]:

# STEP 5: Final preprocessing of the SOS urgency dataset
# This cell only cleans and prepares the existing dataset.
# It does not train a model or create additional SOS scenarios.

from pathlib import Path
import pandas as pd

# --------------------------------------------------
# 1. Load the existing processed dataset
# --------------------------------------------------

input_file = Path.cwd() / "sos_urgency_dataset_processed.csv"

if not input_file.exists():
    raise FileNotFoundError(
        f"Dataset not found: {input_file.resolve()}"
    )

df = pd.read_csv(input_file, low_memory=False)

print("Dataset loaded:", df.shape)


# --------------------------------------------------
# 2. Remove duplicate SOS IDs
# --------------------------------------------------

# Keep the first occurrence of each SOS ID.
before = len(df)

df = df.drop_duplicates(
    subset=["sos_id"],
    keep="first"
).copy()

print("Duplicate-ID rows removed:", before - len(df))


# --------------------------------------------------
# 3. Clean text columns
# --------------------------------------------------

# Remove leading/trailing spaces from text values.
# Do not change the capitalization of descriptive text.
text_columns = df.select_dtypes(
    include=["object", "string"]
).columns

for column in text_columns:
    df[column] = df[column].apply(
        lambda value: value.strip()
        if isinstance(value, str)
        else value
    )

# Convert empty text strings to missing values.
df[text_columns] = df[text_columns].replace(
    r"^\s*$",
    pd.NA,
    regex=True
)


# --------------------------------------------------
# 4. Standardize key categorical columns
# --------------------------------------------------

# Standardize Yes/No values in SOS features.
yes_no_columns = [
    "people_trapped",
    "injuries_reported",
    "medical_emergency",
    "rescue_required",
    "food_required",
    "water_required",
    "shelter_required",
]

for column in yes_no_columns:
    if column in df.columns:
        df[column] = (
            df[column]
            .astype("string")
            .str.strip()
            .str.title()
        )

# Standardize urgency labels.
if "urgency_label" in df.columns:
    df["urgency_label"] = (
        df["urgency_label"]
        .astype("string")
        .str.strip()
        .str.title()
    )

# Standardize other categorical SOS features.
categorical_columns = [
    "water_level",
    "building_condition",
    "location_risk",
]

for column in categorical_columns:
    if column in df.columns:
        df[column] = (
            df[column]
            .astype("string")
            .str.strip()
        )


# --------------------------------------------------
# 5. Check required fields and invalid labels
# --------------------------------------------------

required_columns = [
    "sos_id",
    "sos_description",
    "people_affected",
    "urgency_label",
]

missing_required_columns = [
    column for column in required_columns
    if column not in df.columns
]

if missing_required_columns:
    raise ValueError(
        f"Required columns are missing: {missing_required_columns}"
    )

valid_labels = ["Low", "Medium", "High", "Critical"]

# Keep records with a known urgency label and SOS ID.
before = len(df)

df = df.dropna(
    subset=["sos_id", "urgency_label"]
).copy()

df = df[
    df["urgency_label"].isin(valid_labels)
].copy()

print("Rows removed for missing IDs/labels or invalid labels:",
      before - len(df))


# --------------------------------------------------
# 6. Convert numeric features to numeric types
# --------------------------------------------------

numeric_columns = [
    "people_affected",
    "context_peak_flood_level_m",
    "context_event_duration_days",
    "context_rainfall_t1d",
    "context_district_fatalities",
    "context_district_injuries",
    "context_district_population",
    "context_percent_flooded_area",
    "context_dfsi",
]

for column in numeric_columns:
    if column in df.columns:
        df[column] = pd.to_numeric(
            df[column],
            errors="coerce"
        )

# Do not invent replacement values for missing numeric data.
# Keep missing contextual values as missing until feature selection.


# --------------------------------------------------
# 7. Check impossible people counts
# --------------------------------------------------

if "people_affected" in df.columns:
    invalid_count = (
        df["people_affected"].notna()
        & (
            (df["people_affected"] < 1)
            | (df["people_affected"] % 1 != 0)
        )
    )

    print("Rows with invalid people_affected values:",
          int(invalid_count.sum()))

    # Set invalid values to missing rather than inventing a count.
    df.loc[invalid_count, "people_affected"] = pd.NA


# --------------------------------------------------
# 8. Add preprocessing provenance
# --------------------------------------------------

df["preprocessing_status"] = "cleaned"

# Preserve existing record_source and label_source columns.
# Synthetic records must remain identifiable as synthetic.


# --------------------------------------------------
# 9. Save the final cleaned dataset
# --------------------------------------------------

output_file = Path.cwd() / "sos_urgency_dataset_preprocessed.csv"

df.to_csv(output_file, index=False)

print("\nFINAL PREPROCESSING REPORT")
print("--------------------------")
print("Final rows:", len(df))
print("Final columns:", len(df.columns))
print("Duplicate SOS IDs:", df["sos_id"].duplicated().sum())
print("Missing SOS IDs:", df["sos_id"].isna().sum())
print("Invalid urgency labels:",
      (~df["urgency_label"].isin(valid_labels)).sum())

print("\nUrgency label distribution:")
display(
    df["urgency_label"]
    .value_counts()
    .reindex(valid_labels, fill_value=0)
)

print("\nMissing values in core columns:")
display(
    df[
        ["sos_id", "sos_description", "people_affected", "urgency_label"]
    ].isna().sum().to_frame("Missing_Count")
)

print("\nSaved cleaned dataset to:")
print(output_file.resolve())


Dataset loaded: (5016, 40)
Duplicate-ID rows removed: 0
Rows removed for missing IDs/labels or invalid labels: 0
Rows with invalid people_affected values: 0

FINAL PREPROCESSING REPORT
--------------------------
Final rows: 5016
Final columns: 41
Duplicate SOS IDs: 0
Missing SOS IDs: 0
Invalid urgency labels: 0

Urgency label distribution:


urgency_label
Low          132
Medium      1305
High        2375
Critical    1204
Name: count, dtype: Int64


Missing values in core columns:


,Missing_Count
sos_id,0
sos_description,0
people_affected,0
urgency_label,0



Saved cleaned dataset to:
C:\Users\ishit\jeevan_link_ml\Data\Processed_data\sos_urgency_dataset\sos_urgency_dataset_preprocessed.csv


In [4]:

# STEP 6: Final preprocessing checks and feature selection
# No model training happens in this cell.

from pathlib import Path
import pandas as pd

# Load the cleaned dataset created in Step 5.
file_path = Path.cwd() / "sos_urgency_dataset_preprocessed.csv"

if not file_path.exists():
    raise FileNotFoundError(
        "Run Step 5 first. The preprocessed CSV was not found."
    )

df = pd.read_csv(file_path, low_memory=False)

# Define the SOS features that can be used as model inputs later.
# Exclude free-text descriptions and independently sampled source context.
feature_columns = [
    "people_affected",
    "people_trapped",
    "injuries_reported",
    "medical_emergency",
    "water_level",
    "building_condition",
    "rescue_required",
    "food_required",
    "water_required",
    "shelter_required",
    "location_risk",
]

target_column = "urgency_label"

# Check that all expected columns exist.
missing_features = [
    column for column in feature_columns
    if column not in df.columns
]

if missing_features:
    raise ValueError(
        f"Missing expected SOS feature columns: {missing_features}"
    )

if target_column not in df.columns:
    raise ValueError(f"Target column '{target_column}' is missing.")

# Display the selected features and target.
print("Dataset shape:", df.shape)

print("\nSOS input features:")
for column in feature_columns:
    print("-", column)

print("\nTarget column:", target_column)

print("\nSelected feature data types:")
display(df[feature_columns].dtypes.to_frame("Data_Type"))

print("\nMissing values in selected features:")
display(df[feature_columns].isna().sum().to_frame("Missing_Count"))

print("\nUrgency labels:")
display(df[target_column].value_counts(dropna=False))

print("\nSample of selected SOS features and target:")
display(df[feature_columns + [target_column]].head(10))

# Save a preprocessing report without training a model.
report = pd.DataFrame({
    "Column": feature_columns + [target_column],
    "Role": ["Input feature"] * len(feature_columns) + ["Prediction target"],
    "Data_Type": [
        str(df[column].dtype)
        for column in feature_columns + [target_column]
    ],
    "Missing_Count": [
        int(df[column].isna().sum())
        for column in feature_columns + [target_column]
    ],
})

report_path = Path.cwd() / "sos_urgency_preprocessing_report.csv"
report.to_csv(report_path, index=False)

print("\nPreprocessing report saved to:")
print(report_path.resolve())
print("\nNo model has been trained.")


Dataset shape: (5016, 41)

SOS input features:
- people_affected
- people_trapped
- injuries_reported
- medical_emergency
- water_level
- building_condition
- rescue_required
- food_required
- water_required
- shelter_required
- location_risk

Target column: urgency_label

Selected feature data types:


,Data_Type
people_affected,float64
people_trapped,str
injuries_reported,str
medical_emergency,str
water_level,str
building_condition,str
rescue_required,str
food_required,str
water_required,str
shelter_required,str



Missing values in selected features:


,Missing_Count
people_affected,0
people_trapped,0
injuries_reported,0
medical_emergency,0
water_level,0
building_condition,0
rescue_required,0
food_required,0
water_required,0
shelter_required,0



Urgency labels:


urgency_label
High        2375
Medium      1305
Critical    1204
Low          132
Name: count, dtype: int64


Sample of selected SOS features and target:


,people_affected,people_trapped,injuries_reported,medical_emergency,water_level,building_condition,rescue_required,food_required,water_required,shelter_required,location_risk,urgency_label
0,4.0,Yes,No,No,Rapidly_Rising,Intact,Yes,No,Yes,No,High,Critical
1,1.0,Yes,Yes,Yes,Rising,Damaged,Yes,No,No,No,High,Critical
2,1.0,Yes,No,No,Rapidly_Rising,Unknown,Yes,No,No,No,High,Critical
3,6.0,Yes,Yes,No,Rising,Damaged,Yes,No,No,Yes,High,Critical
4,5.0,No,No,No,Rising,Intact,Yes,No,Yes,Yes,High,High
5,1.0,No,Yes,Yes,Stable,Intact,Yes,No,No,No,Medium,High
6,4.0,No,No,No,Rising,Intact,Yes,No,Yes,No,High,High
7,3.0,No,No,No,Stable,Damaged,No,No,No,Yes,High,High
8,4.0,No,No,No,Stable,Intact,No,No,Yes,No,Medium,Medium
9,2.0,No,No,No,Stable,Intact,No,No,No,Yes,Medium,Medium



Preprocessing report saved to:
C:\Users\ishit\jeevan_link_ml\Data\Processed_data\sos_urgency_dataset\sos_urgency_preprocessing_report.csv

No model has been trained.


In [5]:

# STEP 7: Encode SOS features for future model training
# This is preprocessing only. No model is trained.

from pathlib import Path
import pandas as pd

# Load the cleaned dataset
input_file = Path.cwd() / "sos_urgency_dataset_preprocessed.csv"

if not input_file.exists():
    raise FileNotFoundError(
        "Run the previous preprocessing steps first."
    )

df = pd.read_csv(input_file, low_memory=False)

# Define the SOS features and prediction target
feature_columns = [
    "people_affected",
    "people_trapped",
    "injuries_reported",
    "medical_emergency",
    "water_level",
    "building_condition",
    "rescue_required",
    "food_required",
    "water_required",
    "shelter_required",
    "location_risk",
]

target_column = "urgency_label"

# Confirm required columns exist
required_columns = feature_columns + [target_column]
missing_columns = [
    column for column in required_columns
    if column not in df.columns
]

if missing_columns:
    raise ValueError(f"Missing columns: {missing_columns}")

# Make a copy so the cleaned dataset stays unchanged
model_data = df[required_columns].copy()

# Convert Yes/No features to 1/0
binary_columns = [
    "people_trapped",
    "injuries_reported",
    "medical_emergency",
    "rescue_required",
    "food_required",
    "water_required",
    "shelter_required",
]

for column in binary_columns:
    model_data[column] = (
        model_data[column]
        .astype("string")
        .str.strip()
        .str.title()
        .map({"Yes": 1, "No": 0})
    )

# Convert people_affected to numeric
model_data["people_affected"] = pd.to_numeric(
    model_data["people_affected"],
    errors="coerce"
)

# One-hot encode the remaining categorical features.
categorical_columns = [
    "water_level",
    "building_condition",
    "location_risk",
]

model_data = pd.get_dummies(
    model_data,
    columns=categorical_columns,
    dummy_na=True,
    dtype=int
)

# Handle missing numeric feature values using the median.
# If a column is entirely missing, fill it with zero.
feature_names = [
    column for column in model_data.columns
    if column != target_column
]

for column in feature_names:
    model_data[column] = pd.to_numeric(
        model_data[column],
        errors="coerce"
    )

    median_value = model_data[column].median()

    if pd.isna(median_value):
        median_value = 0

    model_data[column] = model_data[column].fillna(median_value)

# Remove records without a valid target label.
valid_labels = ["Low", "Medium", "High", "Critical"]

before = len(model_data)

model_data = model_data[
    model_data[target_column].isin(valid_labels)
].copy()

print("Rows removed due to invalid/missing target:",
      before - len(model_data))

# Save the encoded dataset separately.
output_file = Path.cwd() / "sos_urgency_dataset_encoded.csv"

model_data.to_csv(output_file, index=False)

print("\nEncoded dataset shape:", model_data.shape)
print("Remaining missing feature values:",
      int(model_data[feature_names].isna().sum().sum()))

print("\nUrgency label distribution:")
display(model_data[target_column].value_counts())

print("\nEncoded feature preview:")
display(model_data.head())

print("\nSaved to:")
print(output_file.resolve())

print("\nNo model training has been performed.")


Rows removed due to invalid/missing target: 0

Encoded dataset shape: (5016, 22)
Remaining missing feature values: 0

Urgency label distribution:


urgency_label
High        2375
Medium      1305
Critical    1204
Low          132
Name: count, dtype: int64


Encoded feature preview:


,people_affected,people_trapped,injuries_reported,medical_emergency,rescue_required,food_required,water_required,shelter_required,urgency_label,water_level_Rapidly_Rising,...,water_level_nan,building_condition_Collapsing,building_condition_Damaged,building_condition_Intact,building_condition_Unknown,building_condition_nan,location_risk_High,location_risk_Low,location_risk_Medium,location_risk_nan
0,4.0,1,0,0,1,0,1,0,Critical,1,...,0,0,0,1,0,0,1,0,0,0
1,1.0,1,1,1,1,0,0,0,Critical,0,...,0,0,1,0,0,0,1,0,0,0
2,1.0,1,0,0,1,0,0,0,Critical,1,...,0,0,0,0,1,0,1,0,0,0
3,6.0,1,1,0,1,0,0,1,Critical,0,...,0,0,1,0,0,0,1,0,0,0
4,5.0,0,0,0,1,0,1,1,High,0,...,0,0,0,1,0,0,1,0,0,0



Saved to:
C:\Users\ishit\jeevan_link_ml\Data\Processed_data\sos_urgency_dataset\sos_urgency_dataset_encoded.csv

No model training has been performed.


In [6]:

# STEP 8: Validate the encoded SOS urgency dataset
# This is the final quality check before the training stage.

from pathlib import Path
import pandas as pd

# Load the encoded dataset created in Step 7
file_path = Path.cwd() / "sos_urgency_dataset_encoded.csv"

if not file_path.exists():
    raise FileNotFoundError(
        "Encoded dataset not found. Run Step 7 first."
    )

df = pd.read_csv(file_path, low_memory=False)

# Define the prediction target and feature columns
target_column = "urgency_label"
feature_columns = [
    column for column in df.columns
    if column != target_column
]

# 1. Check dataset dimensions
print("Total rows:", len(df))
print("Total columns:", len(df.columns))
print("Feature count:", len(feature_columns))

# 2. Check missing values
print("\nMissing values by column:")
display(df.isna().sum().to_frame("Missing_Count"))

print(
    "\nTotal missing feature values:",
    int(df[feature_columns].isna().sum().sum())
)

# 3. Check whether every feature is numeric
non_numeric_columns = [
    column for column in feature_columns
    if not pd.api.types.is_numeric_dtype(df[column])
]

print("\nNon-numeric feature columns:", non_numeric_columns)

# 4. Check valid target labels
valid_labels = ["Low", "Medium", "High", "Critical"]

print("\nUrgency label counts:")
display(
    df[target_column]
    .value_counts(dropna=False)
    .reindex(valid_labels, fill_value=0)
)

invalid_label_count = int(
    (~df[target_column].isin(valid_labels)).sum()
)

print("Invalid or missing target labels:", invalid_label_count)

# 5. Check duplicate rows
print("\nDuplicate complete rows:", df.duplicated().sum())

# 6. Save a validation report
report = pd.DataFrame({
    "Check": [
        "Total rows",
        "Total columns",
        "Feature count",
        "Missing feature values",
        "Non-numeric feature columns",
        "Invalid or missing target labels",
        "Duplicate complete rows",
    ],
    "Result": [
        len(df),
        len(df.columns),
        len(feature_columns),
        int(df[feature_columns].isna().sum().sum()),
        len(non_numeric_columns),
        invalid_label_count,
        int(df.duplicated().sum()),
    ],
})

report_file = Path.cwd() / "sos_urgency_final_validation_report.csv"
report.to_csv(report_file, index=False)

print("\nValidation report saved to:")
print(report_file.resolve())


Total rows: 5016
Total columns: 22
Feature count: 21

Missing values by column:


,Missing_Count
people_affected,0
people_trapped,0
injuries_reported,0
medical_emergency,0
rescue_required,0
food_required,0
water_required,0
shelter_required,0
urgency_label,0
water_level_Rapidly_Rising,0



Total missing feature values: 0

Non-numeric feature columns: []

Urgency label counts:


urgency_label
Low          132
Medium      1305
High        2375
Critical    1204
Name: count, dtype: int64

Invalid or missing target labels: 0

Duplicate complete rows: 862

Validation report saved to:
C:\Users\ishit\jeevan_link_ml\Data\Processed_data\sos_urgency_dataset\sos_urgency_final_validation_report.csv


In [7]:

# STEP 9: Create a final preprocessing summary
# This step only documents the prepared datasets.
# No model training is performed.

from pathlib import Path
import pandas as pd

# Files produced during preprocessing
base_dir = Path.cwd()

files_to_check = [
    "sos_urgency_data_collection.csv",
    "sos_urgency_dataset_processed.csv",
    "sos_urgency_dataset_preprocessed.csv",
    "sos_urgency_dataset_encoded.csv",
    "sos_urgency_final_validation_report.csv",
]

summary_rows = []

for filename in files_to_check:
    file_path = base_dir / filename

    if file_path.exists():
        # Read the CSV to confirm its current dimensions.
        data = pd.read_csv(file_path, low_memory=False)

        summary_rows.append({
            "File": filename,
            "Exists": "Yes",
            "Rows": data.shape[0],
            "Columns": data.shape[1],
            "File_Size_KB": round(file_path.stat().st_size / 1024, 2),
        })
    else:
        summary_rows.append({
            "File": filename,
            "Exists": "No",
            "Rows": None,
            "Columns": None,
            "File_Size_KB": None,
        })

summary_df = pd.DataFrame(summary_rows)

print("JEEVAN LINK — SOS DATASET PREPROCESSING SUMMARY")
display(summary_df)

# Save the summary for your project documentation.
summary_file = base_dir / "sos_urgency_preprocessing_summary.csv"
summary_df.to_csv(summary_file, index=False)

print("\nSummary saved to:")
print(summary_file.resolve())

print("\nPreprocessing summary completed. No model was trained.")


JEEVAN LINK — SOS DATASET PREPROCESSING SUMMARY


,File,Exists,Rows,Columns,File_Size_KB
0,sos_urgency_data_collection.csv,Yes,16,15,2.26
1,sos_urgency_dataset_processed.csv,Yes,5016,40,2699.46
2,sos_urgency_dataset_preprocessed.csv,Yes,5016,41,2748.47
3,sos_urgency_dataset_encoded.csv,Yes,5016,22,255.85
4,sos_urgency_final_validation_report.csv,Yes,7,2,0.18



Summary saved to:
C:\Users\ishit\jeevan_link_ml\Data\Processed_data\sos_urgency_dataset\sos_urgency_preprocessing_summary.csv

Preprocessing summary completed. No model was trained.


In [8]:

# ============================================================
# JEEVAN LINK: FINAL SOS URGENCY DATASET CLEANING
# ============================================================
# Purpose:
# 1. Load the existing encoded dataset.
# 2. Validate the urgency labels.
# 3. Remove exact duplicate feature/label records.
# 4. Remove constant columns that provide no information.
# 5. Validate numeric features and missing values.
# 6. Save the final cleaned dataset and a cleaning report.
#
# IMPORTANT:
# - This cell does NOT train a model.
# - Synthetic labels remain synthetic.
# - It does not invent real SOS observations.
# - It does not artificially balance the classes.
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# STEP 1: Locate and load the encoded dataset
# ------------------------------------------------------------

base_dir = Path.cwd()

input_file = base_dir / "sos_urgency_dataset_encoded.csv"

if not input_file.exists():
    raise FileNotFoundError(
        f"Could not find:\n{input_file.resolve()}\n\n"
        "Make sure this notebook is in the sos_urgency_dataset "
        "folder and Step 7 has created the encoded CSV."
    )

df = pd.read_csv(input_file, low_memory=False)

print("INPUT DATASET")
print("File:", input_file.name)
print("Rows:", len(df))
print("Columns:", len(df.columns))


# ------------------------------------------------------------
# STEP 2: Validate the target column
# ------------------------------------------------------------

target_column = "urgency_label"

if target_column not in df.columns:
    raise ValueError(
        f"Target column '{target_column}' is missing."
    )

valid_labels = ["Low", "Medium", "High", "Critical"]

# Standardize urgency label text.
df[target_column] = (
    df[target_column]
    .astype("string")
    .str.strip()
    .str.title()
)

# Remove records with missing or invalid target labels.
rows_before = len(df)

df = df[
    df[target_column].isin(valid_labels)
].copy()

invalid_target_rows_removed = rows_before - len(df)

print("\nRows removed due to invalid/missing labels:",
      invalid_target_rows_removed)


# ------------------------------------------------------------
# STEP 3: Check and clean input features
# ------------------------------------------------------------

# The encoded dataset should contain numeric input features.
# All columns other than the target are treated as candidate features.
feature_columns = [
    column for column in df.columns
    if column != target_column
]

if not feature_columns:
    raise ValueError("No input feature columns were found.")

# Convert candidate features to numeric values.
# Unexpected non-numeric values become missing and are reported below.
for column in feature_columns:
    df[column] = pd.to_numeric(
        df[column],
        errors="coerce"
    )

# Remove feature columns that contain no usable values.
all_missing_columns = [
    column for column in feature_columns
    if df[column].isna().all()
]

if all_missing_columns:
    df = df.drop(columns=all_missing_columns)

print("\nAll-missing columns removed:", all_missing_columns)


# ------------------------------------------------------------
# STEP 4: Remove exact duplicate records
# ------------------------------------------------------------

# Remove rows only when their feature values AND urgency labels
# are identical. This does not guarantee that non-identical
# records represent independent real-world SOS cases.

rows_before = len(df)

df = df.drop_duplicates(
    subset=[
        column for column in df.columns
        if column != target_column
    ] + [target_column],
    keep="first"
).reset_index(drop=True)

duplicate_rows_removed = rows_before - len(df)

print("Exact duplicate feature/label rows removed:",
      duplicate_rows_removed)


# ------------------------------------------------------------
# STEP 5: Remove constant feature columns
# ------------------------------------------------------------

# A constant feature has the same value in every remaining row.
# It cannot help distinguish urgency classes.

feature_columns = [
    column for column in df.columns
    if column != target_column
]

constant_columns = [
    column for column in feature_columns
    if df[column].nunique(dropna=False) <= 1
]

df = df.drop(columns=constant_columns)

print("Constant feature columns removed:", constant_columns)


# ------------------------------------------------------------
# STEP 6: Check missing values
# ------------------------------------------------------------

feature_columns = [
    column for column in df.columns
    if column != target_column
]

missing_counts = df[feature_columns].isna().sum()

print("\nMissing values in each feature:")
display(missing_counts.to_frame("Missing_Count"))

total_missing = int(missing_counts.sum())

if total_missing > 0:
    # Use median imputation for remaining numeric features.
    # This is a prepared file, not a fitted training pipeline.
    # For model evaluation, fit imputation on training data only.
    for column in feature_columns:
        if df[column].isna().any():
            median_value = df[column].median()

            if pd.isna(median_value):
                raise ValueError(
                    f"Cannot safely impute column: {column}"
                )

            df[column] = df[column].fillna(median_value)

print("Missing feature values after cleaning:",
      int(df[feature_columns].isna().sum().sum()))


# ------------------------------------------------------------
# STEP 7: Validate feature values and labels
# ------------------------------------------------------------

# Check that all input features are numeric.
non_numeric_columns = [
    column for column in feature_columns
    if not pd.api.types.is_numeric_dtype(df[column])
]

if non_numeric_columns:
    raise ValueError(
        f"Non-numeric features remain: {non_numeric_columns}"
    )

# Check for infinite values.
infinite_counts = np.isinf(
    df[feature_columns].to_numpy(dtype=float)
).sum()

if infinite_counts > 0:
    raise ValueError(
        "Infinite feature values remain. Inspect the affected columns."
    )

# Verify labels are valid.
invalid_labels_remaining = int(
    (~df[target_column].isin(valid_labels)).sum()
)

if invalid_labels_remaining > 0:
    raise ValueError("Invalid urgency labels remain after cleaning.")

# Verify feature matrix is not empty.
if df.empty or not feature_columns:
    raise ValueError(
        "The dataset has no usable records or features."
    )


# ------------------------------------------------------------
# STEP 8: Review the class distribution
# ------------------------------------------------------------

class_counts = (
    df[target_column]
    .value_counts()
    .reindex(valid_labels, fill_value=0)
)

print("\nURGENCY CLASS DISTRIBUTION")
display(class_counts.to_frame("Record_Count"))

print(
    "\nNote: classes are reported as-is. "
    "They have not been artificially balanced."
)


# ------------------------------------------------------------
# STEP 9: Save the final cleaned dataset
# ------------------------------------------------------------

output_file = base_dir / "sos_urgency_dataset_final_cleaned.csv"

df.to_csv(output_file, index=False)

# Save a report documenting the cleaning operations.
report_rows = [
    {"Check": "Input rows", "Result": len(pd.read_csv(input_file, low_memory=False))},
    {"Check": "Final rows", "Result": len(df)},
    {"Check": "Final columns including target", "Result": len(df.columns)},
    {"Check": "Final feature count", "Result": len(feature_columns)},
    {"Check": "Invalid target rows removed", "Result": invalid_target_rows_removed},
    {"Check": "Exact duplicate rows removed", "Result": duplicate_rows_removed},
    {"Check": "Constant columns removed", "Result": len(constant_columns)},
    {"Check": "All-missing columns removed", "Result": len(all_missing_columns)},
    {"Check": "Missing feature values remaining", "Result": int(df[feature_columns].isna().sum().sum())},
    {"Check": "Invalid urgency labels remaining", "Result": invalid_labels_remaining},
    {"Check": "Non-numeric feature columns remaining", "Result": len(non_numeric_columns)},
]

report_df = pd.DataFrame(report_rows)

report_file = base_dir / "sos_urgency_final_cleaning_report.csv"
report_df.to_csv(report_file, index=False)


# ------------------------------------------------------------
# STEP 10: Display final results
# ------------------------------------------------------------

print("\n" + "=" * 55)
print("FINAL SOS URGENCY DATASET CLEANING REPORT")
print("=" * 55)

display(report_df)

print("\nFinal dataset shape:", df.shape)
print("Final feature count:", len(feature_columns))
print("Duplicate feature/label rows:", df.duplicated().sum())
print("Missing feature values:",
      int(df[feature_columns].isna().sum().sum()))
print("Invalid urgency labels:",
      int((~df[target_column].isin(valid_labels)).sum()))

print("\nFinal dataset preview:")
display(df.head())

print("\nFinal dataset saved to:")
print(output_file.resolve())

print("\nCleaning report saved to:")
print(report_file.resolve())

print("\nPreprocessing completed. No model was trained.")


INPUT DATASET
File: sos_urgency_dataset_encoded.csv
Rows: 5016
Columns: 22

Rows removed due to invalid/missing labels: 0

All-missing columns removed: []
Exact duplicate feature/label rows removed: 862
Constant feature columns removed: ['water_level_nan', 'building_condition_nan', 'location_risk_nan']

Missing values in each feature:


,Missing_Count
people_affected,0
people_trapped,0
injuries_reported,0
medical_emergency,0
rescue_required,0
food_required,0
water_required,0
shelter_required,0
water_level_Rapidly_Rising,0
water_level_Rising,0


Missing feature values after cleaning: 0

URGENCY CLASS DISTRIBUTION


,Record_Count
urgency_label,
Low,85
Medium,945
High,1989
Critical,1135



Note: classes are reported as-is. They have not been artificially balanced.

FINAL SOS URGENCY DATASET CLEANING REPORT


,Check,Result
0,Input rows,5016
1,Final rows,4154
2,Final columns including target,19
3,Final feature count,18
4,Invalid target rows removed,0
5,Exact duplicate rows removed,862
6,Constant columns removed,3
7,All-missing columns removed,0
8,Missing feature values remaining,0
9,Invalid urgency labels remaining,0



Final dataset shape: (4154, 19)
Final feature count: 18
Duplicate feature/label rows: 0
Missing feature values: 0
Invalid urgency labels: 0

Final dataset preview:


,people_affected,people_trapped,injuries_reported,medical_emergency,rescue_required,food_required,water_required,shelter_required,urgency_label,water_level_Rapidly_Rising,water_level_Rising,water_level_Stable,building_condition_Collapsing,building_condition_Damaged,building_condition_Intact,building_condition_Unknown,location_risk_High,location_risk_Low,location_risk_Medium
0,4.0,1,0,0,1,0,1,0,Critical,1,0,0,0,0,1,0,1,0,0
1,1.0,1,1,1,1,0,0,0,Critical,0,1,0,0,1,0,0,1,0,0
2,1.0,1,0,0,1,0,0,0,Critical,1,0,0,0,0,0,1,1,0,0
3,6.0,1,1,0,1,0,0,1,Critical,0,1,0,0,1,0,0,1,0,0
4,5.0,0,0,0,1,0,1,1,High,0,1,0,0,0,1,0,1,0,0



Final dataset saved to:
C:\Users\ishit\jeevan_link_ml\Data\Processed_data\sos_urgency_dataset\sos_urgency_dataset_final_cleaned.csv

Cleaning report saved to:
C:\Users\ishit\jeevan_link_ml\Data\Processed_data\sos_urgency_dataset\sos_urgency_final_cleaning_report.csv

Preprocessing completed. No model was trained.
